In [1]:
# %% Cell 1: Imports
import os, glob, json, time, warnings, gc, random, re
from collections import defaultdict
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

try:
    import rasterio
    HAS_RASTERIO = True
except ImportError:
    HAS_RASTERIO = False
    import tifffile

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}, "
          f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f}GB")

PIPELINE_START = time.time()

GPU: Tesla T4, VRAM: 15.6GB


In [2]:
# %% Cell 2: Config (FP-specific exclusions from diagnostic audit)
class CFG:
    # Paths
    DATA_ROOT = ""
    for p in ["/kaggle/input/ts-satfire/ts-satfire/",
              "/kaggle/input/datasets/z789456sx/ts-satfire/ts-satfire/",
              "/kaggle/input/ts-satfire/ts-satfire/ts-satfire/",
              "/kaggle/input/datasets/z789456sx/ts-satfire/ts-satfire/ts-satfire/"]:
        if os.path.exists(p):
            DATA_ROOT = p
            break
    SAVE_DIR = "/kaggle/working/"

    # Data
    SEED = 44
    TS_LENGTH = 2
    CROP_SIZE = 256
    # 8 VIIRS (6 day + 2 night) + 18 FirePred aux + 1 cumulative BA
    N_CHANNELS = 27

    # VIIRS normalization 
    SAT_MEAN = np.array([18.76488, 27.441864, 20.584806, 305.99478,
                         294.31738, 14.625097, 276.4207, 275.16766], dtype=np.float32)
    SAT_STD  = np.array([15.911591, 14.879259, 10.832616, 21.761852,
                         24.703484, 9.878246, 40.64329, 40.7657], dtype=np.float32)

    # FirePred band selection (skip band 3 - all zeros in probe fire)
    # 0-indexed indices: keep 0,1,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18 -> 18 bands
    FP_BAND_IDX = [0, 1, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]
    # Normalization stats for FirePred computed from training sample at preload time
    FP_MEAN = None
    FP_STD = None

    # Model
    ENCODER_CHANNELS = [64, 128, 256, 512]
    SE_REDUCTION = 8

    # Training
    BATCH_SIZE = 8
    NUM_WORKERS = 0
    LR = 5e-4
    WEIGHT_DECAY = 1e-4
    EPOCHS = 60
    USE_AMP = True

    # Loss
    POS_WEIGHT = 300.0
    FOCAL_GAMMA = 3.0
    FOCAL_ALPHA = 0.85
    DICE_WEIGHT = 0.5
    FOCAL_WEIGHT = 0.3
    BCE_WEIGHT = 0.2

    # Sampling
    MIN_BURN_PX = 3
    MAX_NEG_RATIO = 1.5
    INTERVAL = 1

    # FP val set (paper, 15 IDs). 23301962 is NOT in the Kaggle release -> silently absent.
    PRED_VAL_IDS = [
        "20568194","20701026","20562846","20700973","24462610",
        "24462788","24462753","24103571","21998313","21751303",
        "22141596","21999381","23301962","22712904","22713339",
    ]

    # FP-specific exclusion lists from fp-label-audit notebook
    FP_TRAIN_EXCLUDE = [
        "20702159","20777134","20777152","20777181","20777195",
        "20777203","20777207","20777386","20777397","20777452",
        "20777459","20777482","20777487","20777508","20777521",
        "20778140","20778153","20778225","20899721","20899766",
        "20899892","20899967","20900040","20900085","20900124",
        "20900131","20901111","20901127","21158900","21158932",
        "21158939","21158940","21693566","21889672","21889683",
        "21889697","21889719","21889734","21889754","21997770",
        "21997775","22712973","23036871","23860939","23860978",
        "23861018","23861131","24332700",
    ]
    FP_VAL_EXCLUDE = [
        "20562846","20568194","20700973","20701026","22712904","22713339",
    ]
    FP_TEST_EXCLUDE = [
        "US_2021_FL2521008104520210308",
        "US_2021_MT4714310953420211004",
        "US_2021_NM3323810847220210520",
        "US_2021_NM3340210587120210426",
        "US_2021_NM3344410803520210514",
        "US_2021_NM3676810505920211120",
        "US_2021_AZ3345510938920210616",
        "US_2021_AZ3368910927620210616",
    ]

random.seed(CFG.SEED); np.random.seed(CFG.SEED); torch.manual_seed(CFG.SEED)
torch.cuda.manual_seed_all(CFG.SEED)

SFX = f"_s{CFG.SEED}"          # appended to every output file
print(f"SEED: {CFG.SEED}  (output suffix '{SFX}')")
print(f"DATA_ROOT: {CFG.DATA_ROOT}")
print(f"N_CHANNELS: {CFG.N_CHANNELS} (8 VIIRS + 18 FirePred + 1 BA)")

SEED: 44  (output suffix '_s44')
DATA_ROOT: /kaggle/input/datasets/z789456sx/ts-satfire/ts-satfire/
N_CHANNELS: 27 (8 VIIRS + 18 FirePred + 1 BA)


In [3]:
# %% Cell 3: Data loading helpers

def read_tif(path):
    if HAS_RASTERIO:
        with rasterio.open(path) as src:
            return src.read().astype(np.float32)
    else:
        arr = tifffile.imread(path).astype(np.float32)
        if arr.ndim == 2: arr = arr[np.newaxis]
        return arr

def load_frame_sat(fire_dir, day_path):
    """Load 8 satellite channels (6 day + 2 night), normalized. Returns (sat, H, W)."""
    d = read_tif(day_path)
    bands = d[:6]
    H, W = bands.shape[1], bands.shape[2]

    night_dir = os.path.join(fire_dir, "VIIRS_Night")
    night_fname = os.path.basename(day_path).replace("_VIIRS_Day", "_VIIRS_Night")
    night_path = os.path.join(night_dir, night_fname)
    if os.path.exists(night_path):
        n = read_tif(night_path)
        nb = n[:2] if n.shape[0] >= 2 else np.zeros((2, H, W), dtype=np.float32)
    else:
        nb = np.zeros((2, H, W), dtype=np.float32)

    sat = np.concatenate([bands, nb], axis=0)
    mean = CFG.SAT_MEAN.reshape(-1, 1, 1)
    std = CFG.SAT_STD.reshape(-1, 1, 1)
    sat = (sat - mean) / (std + 1e-8)
    sat = np.nan_to_num(sat, nan=0.0, posinf=0.0, neginf=0.0)
    return sat, H, W

def load_firepred_raw(fire_dir, day_path):
    """Load FirePred 18-band feature stack (selected bands). Returns (18, Hf, Wf) or None.
    FirePred may have different spatial dims than VIIRS_Day (e.g. 447x447 vs 596x595).
    Callers center-crop each independently; FP's geographic center is assumed to align
    with VIIRS_Day's within the 256x256 patch.
    """
    fp_dir = os.path.join(fire_dir, "FirePred")
    fp_fname = os.path.basename(day_path).replace("_VIIRS_Day", "_FirePred")
    fp_path = os.path.join(fp_dir, fp_fname)
    if not os.path.exists(fp_path):
        return None
    arr = read_tif(fp_path)
    if arr.shape[0] < 19:
        return None
    return arr[CFG.FP_BAND_IDX]

def load_ba_mask(day_path):
    """Load cumulative BA mask from VIIRS_Day band 8 (0-indexed: 7). Returns (H, W) or None."""
    d = read_tif(day_path)
    if d.shape[0] >= 8:
        return np.isfinite(d[7]).astype(np.float32)
    return None

def center_crop(arr, cs, H, W):
    r0 = max(0, (H - cs) // 2)
    c0 = max(0, (W - cs) // 2)
    if arr.ndim == 3:
        return arr[:, r0:r0+cs, c0:c0+cs]
    elif arr.ndim == 2:
        return arr[r0:r0+cs, c0:c0+cs]
    raise ValueError(f"Unexpected ndim={arr.ndim}")


def compute_firepred_stats(fire_infos, exclude_ids, max_fires=20, max_days_per_fire=10):
    """Compute per-channel mean/std of FirePred bands over a subset of training fires.
    Returns (mean, std) each shape (18,).
    """
    print(f"Computing FirePred stats from up to {max_fires} train fires...")
    n_ch = len(CFG.FP_BAND_IDX)
    sums = np.zeros(n_ch, dtype=np.float64)
    sqsums = np.zeros(n_ch, dtype=np.float64)
    counts = np.zeros(n_ch, dtype=np.int64)
    sampled = 0
    for info in fire_infos:
        if sampled >= max_fires: break
        if info["fire_id"] in exclude_ids: continue
        days = info["day_files"]
        step = max(1, len(days) // max_days_per_fire)
        used_any = False
        for day_path in days[::step][:max_days_per_fire]:
            fp = load_firepred_raw(info["fire_dir"], day_path)
            if fp is None: continue
            used_any = True
            fp = np.nan_to_num(fp, nan=0.0, posinf=0.0, neginf=0.0)
            for c in range(n_ch):
                ch = fp[c]
                sums[c]   += ch.sum()
                sqsums[c] += (ch * ch).sum()
                counts[c] += ch.size
        if used_any: sampled += 1
    mean = (sums / np.maximum(counts, 1)).astype(np.float32)
    var  = (sqsums / np.maximum(counts, 1)) - mean.astype(np.float64) ** 2
    std  = np.sqrt(np.maximum(var, 1e-6)).astype(np.float32)
    print(f"  FP mean range: [{mean.min():.3f}, {mean.max():.3f}]")
    print(f"  FP std  range: [{std.min():.3f}, {std.max():.3f}]")
    return mean, std


def normalize_firepred(fp, mean, std):
    m = mean.reshape(-1, 1, 1)
    s = np.maximum(std, 0.1).reshape(-1, 1, 1)   # floor std to avoid blow-up
    out = (fp - m) / s
    out = np.clip(out, -5.0, 5.0)                 # clip extreme z-scores
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)

In [4]:
# %% Cell 4: Preload all data to RAM

def preload_split(fire_infos, split_name, exclude_ids, is_train=False):
    cs = CFG.CROP_SIZE
    ts = CFG.TS_LENGTH
    rng = random.Random(CFG.SEED)
    windows = []
    n_pos = n_neg = n_neg_kept = n_skip_label = n_skip_fp = 0

    for fi, info in enumerate(fire_infos):
        fire_id = info["fire_id"]
        if fire_id in exclude_ids:
            continue
        fire_dir = info["fire_dir"]
        day_files = info["day_files"]
        if len(day_files) < ts + 1:
            continue

        for i in range(0, len(day_files) - ts, CFG.INTERVAL):
            input_days = day_files[i:i+ts]
            label_day = day_files[i+ts]

            ba_last = load_ba_mask(input_days[-1])
            ba_next = load_ba_mask(label_day)
            if ba_last is None or ba_next is None:
                n_skip_label += 1
                continue

            H, W = ba_last.shape
            if H < cs or W < cs:
                n_skip_label += 1
                continue

            label = np.clip(ba_next - ba_last, 0, 1)
            label_crop = center_crop(label, cs, H, W)
            n_burn = label_crop.sum()
            is_positive = n_burn >= CFG.MIN_BURN_PX

            if is_positive:
                n_pos += 1
            else:
                n_neg += 1
                if is_train:
                    if rng.random() > 1.0 / (CFG.MAX_NEG_RATIO + 1):
                        continue
                n_neg_kept += 1

            # Load per-day frames: 8 sat + 18 FirePred + 1 BA = 27 channels
            frames = []
            valid = True
            for t_idx in range(ts):
                sat, fH, fW = load_frame_sat(fire_dir, input_days[t_idx])
                if fH < cs or fW < cs:
                    valid = False; break
                sat_crop = center_crop(sat, cs, fH, fW)  # (8, 256, 256)

                # FirePred 18-band stack
                fp_raw = load_firepred_raw(fire_dir, input_days[t_idx])
                if fp_raw is None:
                    fp_crop = np.zeros((len(CFG.FP_BAND_IDX), cs, cs), dtype=np.float32)
                    n_skip_fp += 1
                else:
                    fp_raw = np.nan_to_num(fp_raw, nan=0.0, posinf=0.0, neginf=0.0)
                    fp_norm = normalize_firepred(fp_raw, CFG.FP_MEAN, CFG.FP_STD)
                    fpH, fpW = fp_norm.shape[1], fp_norm.shape[2]
                    if fpH < cs or fpW < cs:
                        fp_crop = np.zeros((len(CFG.FP_BAND_IDX), cs, cs), dtype=np.float32)
                        n_skip_fp += 1
                    else:
                        fp_crop = center_crop(fp_norm, cs, fpH, fpW)  # (18, 256, 256)

                # Cumulative BA at this input day
                if t_idx == ts - 1:
                    ba_t = ba_last
                else:
                    ba_t = load_ba_mask(input_days[t_idx])
                if ba_t is not None:
                    ba_crop = center_crop(ba_t, cs, fH, fW)[np.newaxis]
                else:
                    ba_crop = np.zeros((1, cs, cs), dtype=np.float32)

                frame = np.concatenate([sat_crop, fp_crop, ba_crop], axis=0)  # (27, 256, 256)
                frames.append(frame)

            if not valid:
                continue

            x = np.stack(frames, axis=1)  # (27, 2, 256, 256)
            windows.append({
                "x": x.astype(np.float16),
                "y": label_crop.astype(np.float16),
                "fire_id": fire_id,
                "n_pos": float(n_burn),
            })

        if (fi + 1) % 20 == 0 or fi + 1 == len(fire_infos):
            mem_mb = sum(w["x"].nbytes + w["y"].nbytes for w in windows) / 1e6
            print(f"\r  {split_name}: {fi+1}/{len(fire_infos)} fires | "
                  f"{len(windows)} windows | {mem_mb:.0f}MB RAM", end="", flush=True)

    print(f"\n  {split_name} done: {len(windows)} windows "
          f"(pos={n_pos}, neg_kept={n_neg_kept}/{n_neg}, "
          f"skip_label={n_skip_label}, skip_fp={n_skip_fp})")
    return windows


def discover_fires():
    all_dirs = sorted([d for d in os.listdir(CFG.DATA_ROOT)
                       if os.path.isdir(os.path.join(CFG.DATA_ROOT, d))])
    train_fires, val_fires, test_fires = [], [], []
    for fire_id in all_dirs:
        fire_dir = os.path.join(CFG.DATA_ROOT, fire_id)
        day_dir = os.path.join(fire_dir, "VIIRS_Day")
        if not os.path.isdir(day_dir):
            continue
        day_files = sorted(glob.glob(os.path.join(day_dir, "*.tif")))
        if len(day_files) < CFG.TS_LENGTH + 1:
            continue
        info = {"fire_id": fire_id, "fire_dir": fire_dir, "day_files": day_files}
        if fire_id.startswith("US_2021_"):
            test_fires.append(info)
        elif fire_id in CFG.PRED_VAL_IDS:
            val_fires.append(info)
        else:
            try:
                int(fire_id); train_fires.append(info)
            except ValueError:
                pass  # Named AF test fires -- not used for FP
    print(f"Discovered: train={len(train_fires)}, val={len(val_fires)}, test={len(test_fires)}")
    return train_fires, val_fires, test_fires


print("Discovering fires...")
train_fires, val_fires, test_fires = discover_fires()

# Compute FirePred normalization stats from training fires before preload
CFG.FP_MEAN, CFG.FP_STD = compute_firepred_stats(train_fires, set(CFG.FP_TRAIN_EXCLUDE))

print("\nPreloading train data...")
train_windows = preload_split(train_fires, "train", set(CFG.FP_TRAIN_EXCLUDE), is_train=True)

print("\nPreloading val data...")
val_windows = preload_split(val_fires, "val", set(CFG.FP_VAL_EXCLUDE), is_train=False)

total_pos = sum(w["n_pos"] for w in train_windows)
total_pix = len(train_windows) * CFG.CROP_SIZE * CFG.CROP_SIZE
pos_ratio = total_pos / (total_pix + 1e-8)
computed_weight = min((1 - pos_ratio) / (pos_ratio + 1e-8), 500.0) if pos_ratio > 0 else 300.0
CFG.POS_WEIGHT = computed_weight
print(f"\nTrain: pos_ratio={pos_ratio:.6f}, POS_WEIGHT={CFG.POS_WEIGHT:.1f}")

train_mem = sum(w["x"].nbytes + w["y"].nbytes for w in train_windows) / 1e6
val_mem = sum(w["x"].nbytes + w["y"].nbytes for w in val_windows) / 1e6
print(f"RAM: train={train_mem:.0f}MB, val={val_mem:.0f}MB")

# Save FP stats alongside weights so viz notebook uses identical normalization
np.savez(os.path.join(CFG.SAVE_DIR, f"fp_stats_v6{SFX}.npz"),
         mean=CFG.FP_MEAN, std=CFG.FP_STD)
print(f"Saved fp_stats_v6.npz: mean range [{CFG.FP_MEAN.min():.3f}, {CFG.FP_MEAN.max():.3f}], std range [{CFG.FP_STD.min():.3f}, {CFG.FP_STD.max():.3f}]")
print(f"Preload done in {(time.time()-PIPELINE_START)/60:.1f}min")

Discovering fires...
Discovered: train=124, val=13, test=24
Computing FirePred stats from up to 20 train fires...
  FP mean range: [-5.454, 4595.527]
  FP std  range: [0.002, 2422.209]

Preloading train data...
  train: 124/124 fires | 1287 windows | 9278MB RAM
  train done: 1287 windows (pos=1117, neg_kept=170/397, skip_label=0, skip_fp=0)

Preloading val data...
  val: 13/13 fires | 167 windows | 1204MB RAM
  val done: 167 windows (pos=121, neg_kept=46/46, skip_label=0, skip_fp=0)

Train: pos_ratio=0.008452, POS_WEIGHT=117.3
RAM: train=9278MB, val=1204MB
Saved fp_stats_v6.npz: mean range [-5.454, 4595.527], std range [0.002, 2422.209]
Preload done in 30.2min


In [5]:
# %% Cell 5: Dataset (RAM-backed)

class PreloadedDataset(Dataset):
    def __init__(self, windows, augment=False):
        self.windows = windows
        self.augment = augment

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        w = self.windows[idx]
        x = w["x"].astype(np.float32)
        y = w["y"].astype(np.float32)
        if self.augment:
            if random.random() > 0.5:
                x = np.flip(x, axis=-1).copy(); y = np.flip(y, axis=-1).copy()
            if random.random() > 0.5:
                x = np.flip(x, axis=-2).copy(); y = np.flip(y, axis=-2).copy()
            k = random.randint(0, 3)
            if k:
                x = np.rot90(x, k, axes=(-2, -1)).copy()
                y = np.rot90(y, k, axes=(0, 1)).copy()
        return torch.from_numpy(x), torch.from_numpy(y).unsqueeze(0)


train_ds = PreloadedDataset(train_windows, augment=True)
val_ds = PreloadedDataset(val_windows, augment=False)
train_dl = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True,
                      num_workers=0, pin_memory=True, drop_last=True)
val_dl = DataLoader(val_ds, batch_size=CFG.BATCH_SIZE, shuffle=False,
                    num_workers=0, pin_memory=True)

xb, yb = next(iter(train_dl))
print(f"Batch: x={tuple(xb.shape)}, y={tuple(yb.shape)}")
print(f"y stats: mean={yb.mean():.6f}, pos%={100*yb.mean():.4f}%")
print(f"Train: {len(train_ds)} samples, {len(train_dl)} batches/epoch")
print(f"Val:   {len(val_ds)} samples, {len(val_dl)} batches/epoch")

Batch: x=(8, 27, 2, 256, 256), y=(8, 1, 256, 256)
y stats: mean=0.014355, pos%=1.4355%
Train: 1287 samples, 160 batches/epoch
Val:   167 samples, 21 batches/epoch


In [6]:
# %% Cell 6: Model (SE-UNet3D, in_ch=27)

class SEBlock3D(nn.Module):
    def __init__(self, ch, r=8):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool3d(1)
        mid = max(ch // r, 4)
        self.fc = nn.Sequential(nn.Linear(ch, mid, bias=False), nn.ReLU(True),
                                nn.Linear(mid, ch, bias=False), nn.Sigmoid())
    def forward(self, x):
        b, c = x.shape[:2]
        return x * self.fc(self.pool(x).view(b, c)).view(b, c, 1, 1, 1)

class ResBlock3D(nn.Module):
    def __init__(self, ic, oc):
        super().__init__()
        self.conv1 = nn.Conv3d(ic, oc, (1,3,3), padding=(0,1,1), bias=False)
        self.bn1 = nn.BatchNorm3d(oc)
        self.conv2 = nn.Conv3d(oc, oc, (1,3,3), padding=(0,1,1), bias=False)
        self.bn2 = nn.BatchNorm3d(oc)
        self.se = SEBlock3D(oc)
        self.skip = nn.Conv3d(ic, oc, 1, bias=False) if ic != oc else nn.Identity()
        self.act = nn.GELU()
    def forward(self, x):
        r = self.skip(x)
        o = self.act(self.bn1(self.conv1(x)))
        o = self.bn2(self.conv2(o))
        return self.act(self.se(o) + r)

class ASPP3D(nn.Module):
    def __init__(self, ic, oc, dils=(1,6,12)):
        super().__init__()
        bc = oc // len(dils)
        self.br = nn.ModuleList([
            nn.Sequential(nn.Conv3d(ic, bc, (1,3,3), padding=(0,d,d),
                          dilation=(1,d,d), bias=False),
                          nn.BatchNorm3d(bc), nn.GELU())
            for d in dils])
        self.gp = nn.Sequential(nn.AdaptiveAvgPool3d((None,1,1)),
                                nn.Conv3d(ic, bc, 1, bias=False),
                                nn.BatchNorm3d(bc), nn.GELU())
        self.fuse = nn.Sequential(nn.Conv3d(bc*(len(dils)+1), oc, 1, bias=False),
                                  nn.BatchNorm3d(oc), nn.GELU())
    def forward(self, x):
        parts = [b(x) for b in self.br]
        g = self.gp(x).expand(-1,-1,x.shape[2],x.shape[3],x.shape[4])
        parts.append(g)
        return self.fuse(torch.cat(parts, 1))

class SEUNet3DPred(nn.Module):
    def __init__(self, in_ch=27, enc=(64,128,256,512), bneck=1024):
        super().__init__()
        self.inp = nn.Sequential(
            nn.Conv3d(in_ch, enc[0], (1,3,3), padding=(0,1,1), bias=False),
            nn.BatchNorm3d(enc[0]), nn.GELU())
        self.encs = nn.ModuleList(); self.pools = nn.ModuleList()
        prev = enc[0]
        for c in enc:
            self.encs.append(ResBlock3D(prev, c))
            self.pools.append(nn.MaxPool3d((1,2,2)))
            prev = c
        self.bneck = ASPP3D(enc[-1], bneck)
        self.ups = nn.ModuleList(); self.decs = nn.ModuleList()
        prev = bneck
        for c in reversed(enc):
            self.ups.append(nn.ConvTranspose3d(prev, c, (1,2,2), stride=(1,2,2)))
            self.decs.append(ResBlock3D(c*2, c))
            prev = c
        self.head = nn.Sequential(
            nn.Conv2d(enc[0], 32, 3, padding=1, bias=False),
            nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32, 1, 1))
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv3d, nn.Conv2d, nn.ConvTranspose3d)):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, (nn.BatchNorm3d, nn.BatchNorm2d)):
                nn.init.constant_(m.weight, 1); nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.inp(x)
        skips = []
        for enc, pool in zip(self.encs, self.pools):
            x = enc(x); skips.append(x); x = pool(x)
        x = self.bneck(x)
        for up, dec, sk in zip(self.ups, self.decs, reversed(skips)):
            x = up(x)
            d = [sk.shape[i]-x.shape[i] for i in range(2,5)]
            if any(di != 0 for di in d):
                x = F.pad(x, [0,d[2],0,d[1],0,d[0]])
            x = dec(torch.cat([x, sk], 1))
        x = x.mean(dim=2)
        return self.head(x)

model = SEUNet3DPred(CFG.N_CHANNELS, CFG.ENCODER_CHANNELS).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {n_params/1e6:.2f}M params")

with torch.no_grad():
    dummy = torch.randn(2, CFG.N_CHANNELS, CFG.TS_LENGTH, CFG.CROP_SIZE, CFG.CROP_SIZE).to(device)
    out = model(dummy)
    print(f"Forward: {tuple(dummy.shape)} -> {tuple(out.shape)}")
    del dummy, out; torch.cuda.empty_cache()

Model: 24.28M params
Forward: (2, 27, 2, 256, 256) -> (2, 1, 256, 256)


In [7]:
# %% Cell 7: Loss

class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__(); self.smooth = smooth
    def forward(self, p, t):
        p = torch.sigmoid(p).view(-1); t = t.view(-1)
        inter = (p * t).sum()
        return 1 - (2*inter + self.smooth) / (p.sum() + t.sum() + self.smooth)

class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.75):
        super().__init__(); self.gamma, self.alpha = gamma, alpha
    def forward(self, p, t):
        bce = F.binary_cross_entropy_with_logits(p, t, reduction="none")
        prob = torch.sigmoid(p)
        pt = prob * t + (1-prob) * (1-t)
        at = self.alpha * t + (1-self.alpha) * (1-t)
        return (at * (1-pt)**self.gamma * bce).mean()

class PredLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.dice = DiceLoss()
        self.focal = FocalLoss(CFG.FOCAL_GAMMA, CFG.FOCAL_ALPHA)
        self.pw = torch.tensor([CFG.POS_WEIGHT]).to(device)
    def forward(self, p, t):
        return (CFG.DICE_WEIGHT * self.dice(p, t) +
                CFG.FOCAL_WEIGHT * self.focal(p, t) +
                CFG.BCE_WEIGHT * F.binary_cross_entropy_with_logits(p, t, pos_weight=self.pw))

criterion = PredLoss()

In [8]:
# %% Cell 8: Training

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=CFG.LR, epochs=CFG.EPOCHS,
    steps_per_epoch=len(train_dl), pct_start=0.1, anneal_strategy="cos")
scaler = GradScaler(enabled=CFG.USE_AMP)

def train_epoch(model, dl):
    model.train()
    tloss, tp, fp, fn = 0, 0, 0, 0
    for xb, yb in dl:
        xb = xb.to(device, non_blocking=True); yb = yb.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=CFG.USE_AMP):
            pred = model(xb); loss = criterion(pred, yb)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update(); scheduler.step()
        tloss += loss.item()
        with torch.no_grad():
            pb = (torch.sigmoid(pred) > 0.5).float()
            tp += (pb * yb).sum().item()
            fp += (pb * (1-yb)).sum().item()
            fn += ((1-pb) * yb).sum().item()
    n = len(dl)
    return tloss/max(n,1), 2*tp/(2*tp+fp+fn+1e-8)

@torch.no_grad()
def validate(model, dl, thr=0.5):
    model.eval()
    tloss, tp, fp, fn = 0, 0, 0, 0
    for xb, yb in dl:
        xb = xb.to(device, non_blocking=True); yb = yb.to(device, non_blocking=True)
        with autocast(enabled=CFG.USE_AMP):
            pred = model(xb); loss = criterion(pred, yb)
        tloss += loss.item()
        pb = (torch.sigmoid(pred) > thr).float()
        tp += (pb * yb).sum().item()
        fp += (pb * (1-yb)).sum().item()
        fn += ((1-pb) * yb).sum().item()
    n = len(dl)
    f1 = 2*tp/(2*tp+fp+fn+1e-8); iou = tp/(tp+fp+fn+1e-8)
    return tloss/max(n,1), f1, iou

print(f"\n{'='*60}\nTRAINING: {len(train_dl)} bat/ep, bs={CFG.BATCH_SIZE}\n{'='*60}")
print(f"{'Ep':>3} {'TrL':>7} {'TrF1':>7} {'VaL':>7} {'VF1':>7} {'VIoU':>7} {'LR':>9} {'T':>5}")
print("-" * 60)

best_f1, best_ep = 0, 0
hist = {"tl":[],"tf":[],"vl":[],"vf":[],"vi":[],"lr":[]}
for ep in range(1, CFG.EPOCHS+1):
    te = time.time()
    if (time.time()-PIPELINE_START)/3600 > 10.5:
        print("Time limit, stopping"); break
    tl, tf = train_epoch(model, train_dl)
    vl, vf, vi = validate(model, val_dl)
    lr = optimizer.param_groups[0]["lr"]
    hist["tl"].append(tl); hist["tf"].append(tf)
    hist["vl"].append(vl); hist["vf"].append(vf); hist["vi"].append(vi); hist["lr"].append(lr)
    elapsed = time.time()-te
    print(f"{ep:3d} {tl:7.4f} {tf:7.4f} {vl:7.4f} {vf:7.4f} {vi:7.4f} {lr:9.6f} {elapsed:5.0f}s", end="")
    if vf > best_f1:
        best_f1, best_ep = vf, ep
        torch.save(model.state_dict(), os.path.join(CFG.SAVE_DIR, f"best_fp_v6{SFX}.pth"))
        print(" *", end="")
    print()
print(f"\nBest val F1: {best_f1:.4f} at epoch {best_ep}")
print(f"Training time: {(time.time()-PIPELINE_START)/60:.1f}min")


TRAINING: 160 bat/ep, bs=8
 Ep     TrL    TrF1     VaL     VF1    VIoU        LR     T
------------------------------------------------------------
  1  0.9094  0.0316  0.6601  0.0690  0.0357  0.000052   141s *
  2  0.6594  0.0958  0.6097  0.0725  0.0376  0.000140   151s *
  3  0.6398  0.1332  0.6144  0.1013  0.0534  0.000260   151s *
  4  0.6389  0.1495  0.5809  0.2862  0.1670  0.000380   151s *
  5  0.6073  0.1795  0.5378  0.2334  0.1321  0.000468   150s
  6  0.5857  0.2481  0.5416  0.1322  0.0708  0.000500   150s
  7  0.5717  0.2684  0.5121  0.1592  0.0865  0.000500   150s
  8  0.5384  0.2931  0.5050  0.3663  0.2242  0.000498   150s *
  9  0.5501  0.3395  0.5347  0.2196  0.1234  0.000496   150s
 10  0.5375  0.3365  0.5303  0.2702  0.1562  0.000493   150s
 11  0.5230  0.3290  0.5050  0.3987  0.2490  0.000489   150s *
 12  0.5305  0.3420  0.5163  0.3572  0.2175  0.000485   150s
 13  0.5172  0.3436  0.5002  0.3908  0.2428  0.000480   150s
 14  0.5113  0.3821  0.5306  0.3934  0.2449  0

In [9]:
# %% Cell 9: Threshold sweep + Test evaluation

model.load_state_dict(torch.load(os.path.join(CFG.SAVE_DIR, f"best_fp_v6{SFX}.pth"),
                                 weights_only=False))
print("\nThreshold sweep:")
best_thr, best_thr_f1 = 0.5, 0
for thr in np.arange(0.05, 0.80, 0.05):
    _, f1, iou = validate(model, val_dl, thr=thr)
    if f1 > best_thr_f1:
        best_thr_f1, best_thr = f1, thr
    print(f"  thr={thr:.2f}: F1={f1:.4f} IoU={iou:.4f}")
print(f"Optimal: thr={best_thr:.2f}, F1={best_thr_f1:.4f}")

print(f"\n{'='*60}\nTEST EVALUATION\n{'='*60}")
print("Preloading test data...")
test_windows = preload_split(test_fires, "test", set(CFG.FP_TEST_EXCLUDE), is_train=False)
test_ds = PreloadedDataset(test_windows, augment=False)
test_dl = DataLoader(test_ds, batch_size=CFG.BATCH_SIZE, shuffle=False,
                     num_workers=0, pin_memory=True)

fire_ids = [w["fire_id"] for w in test_windows]
fire_met = defaultdict(lambda: {"tp":0,"fp":0,"fn":0,"n":0})
model.eval()
sidx = 0
with torch.no_grad():
    for xb, yb in test_dl:
        xb = xb.to(device, non_blocking=True); yb = yb.to(device, non_blocking=True)
        with autocast(enabled=CFG.USE_AMP):
            pred = model(xb)
        pb = (torch.sigmoid(pred) > best_thr).float()
        bs = xb.shape[0]
        for j in range(bs):
            fid = fire_ids[sidx+j] if sidx+j < len(fire_ids) else "unk"
            fire_met[fid]["tp"] += (pb[j]*yb[j]).sum().item()
            fire_met[fid]["fp"] += (pb[j]*(1-yb[j])).sum().item()
            fire_met[fid]["fn"] += ((1-pb[j])*yb[j]).sum().item()
            fire_met[fid]["n"] += 1
        sidx += bs

print(f"\n{'Fire':<50} {'F1':>8} {'IoU':>8} {'N':>5}")
print("-"*75)
atp, afp, afn = 0, 0, 0
for fid in sorted(fire_met.keys()):
    m = fire_met[fid]
    f1 = 2*m["tp"]/(2*m["tp"]+m["fp"]+m["fn"]+1e-8)
    iou = m["tp"]/(m["tp"]+m["fp"]+m["fn"]+1e-8)
    print(f"{fid:<50} {f1:8.4f} {iou:8.4f} {m['n']:5d}")
    atp += m["tp"]; afp += m["fp"]; afn += m["fn"]
agg_f1 = 2*atp/(2*atp+afp+afn+1e-8)
agg_iou = atp/(atp+afp+afn+1e-8)
print("-"*75)
print(f"{'AGGREGATE':<50} {agg_f1:8.4f} {agg_iou:8.4f}")


Threshold sweep:
  thr=0.05: F1=0.3416 IoU=0.2060
  thr=0.10: F1=0.3804 IoU=0.2349
  thr=0.15: F1=0.4003 IoU=0.2502
  thr=0.20: F1=0.4140 IoU=0.2610
  thr=0.25: F1=0.4237 IoU=0.2688
  thr=0.30: F1=0.4312 IoU=0.2749
  thr=0.35: F1=0.4374 IoU=0.2799
  thr=0.40: F1=0.4428 IoU=0.2844
  thr=0.45: F1=0.4474 IoU=0.2881
  thr=0.50: F1=0.4518 IoU=0.2918
  thr=0.55: F1=0.4563 IoU=0.2956
  thr=0.60: F1=0.4601 IoU=0.2988
  thr=0.65: F1=0.4639 IoU=0.3020
  thr=0.70: F1=0.4669 IoU=0.3046
  thr=0.75: F1=0.4700 IoU=0.3072
Optimal: thr=0.75, F1=0.4700

TEST EVALUATION
Preloading test data...
  test: 24/24 fires | 642 windows | 4628MB RAM
  test done: 642 windows (pos=424, neg_kept=218/218, skip_label=0, skip_fp=0)

Fire                                                     F1      IoU     N
---------------------------------------------------------------------------
US_2021_CA3451712013120211011                        0.3656   0.2237    10
US_2021_CA3568711855020210818                        0.4376   0.2

In [10]:
# %% Cell 10: Comparison + plots + save

paper = {
    "U-Net-3D (TS=6)": (0.375, 0.338),
    "SwinUNETR-3D (TS=6)": (0.374, 0.331),
    "UNETR-3D (TS=6)": (0.371, 0.336),
    "Att-U-Net-3D (TS=6)": (0.354, 0.312),
    "SwinUNETR-3D (TS=2)": (0.366, 0.321),
}
best_paper = max(f for f, _ in paper.values())
delta = agg_f1 - best_paper

print(f"\n{'Model':<35} {'F1':>8} {'IoU':>8}")
print("-"*55)
for n, (f, i) in paper.items():
    print(f"{n:<35} {f:8.3f} {i:8.3f}")
print("-"*55)
print(f"{'Ours (TS='+str(CFG.TS_LENGTH)+')':<35} {agg_f1:8.3f} {agg_iou:8.3f}")
print(f"\nDelta: {delta:+.3f} ({'BEATS' if delta > 0 else 'below'} benchmark)")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(hist["tl"], label="Train"); axes[0].plot(hist["vl"], label="Val")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].plot(hist["tf"], label="Train"); axes[1].plot(hist["vf"], label="Val")
axes[1].axhline(best_paper, color="r", ls="--", label=f"Paper({best_paper})")
axes[1].set_title("F1"); axes[1].legend(); axes[1].grid(True, alpha=0.3)
axes[2].plot(hist["vi"], label="Val")
axes[2].axhline(0.338, color="r", ls="--", label="Paper(0.338)")
axes[2].set_title("IoU"); axes[2].legend(); axes[2].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CFG.SAVE_DIR, f"fp_v6_curves{SFX}.png"), dpi=150, bbox_inches="tight")
plt.close()

fig, ax = plt.subplots(figsize=(10, 5))
names = list(paper.keys()) + [f"Ours (TS={CFG.TS_LENGTH})"]
f1s = [f for f, _ in paper.values()] + [agg_f1]
colors = ["#4a86c8"]*len(paper) + ["#e74c3c"]
bars = ax.bar(range(len(names)), f1s, color=colors)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=45, ha="right", fontsize=9)
ax.set_ylabel("F1"); ax.set_title("Fire Prediction Comparison")
ax.grid(True, alpha=0.3, axis="y")
for bar, v in zip(bars, f1s):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.003,
            f"{v:.3f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(CFG.SAVE_DIR, f"fp_v6_compare{SFX}.png"), dpi=150, bbox_inches="tight")
plt.close()

res = {"model":"SE-UNet3D-Pred-v6", "ts":CFG.TS_LENGTH,
       "n_channels":CFG.N_CHANNELS,
       "params_M":n_params/1e6, "best_val_f1":best_f1, "best_epoch":best_ep,
       "threshold":float(best_thr), "test_f1":agg_f1, "test_iou":agg_iou,
       "paper_best_f1":best_paper, "delta":delta,
       "pos_weight":CFG.POS_WEIGHT,
       "train_windows":len(train_windows), "val_windows":len(val_windows),
       "test_windows":len(test_windows)}
with open(os.path.join(CFG.SAVE_DIR, f"fp_v6_results{SFX}.json"), "w") as f:
    json.dump(res, f, indent=2, default=float)

print(f"\nTotal pipeline: {(time.time()-PIPELINE_START)/60:.1f}min")
print("Done.")


Model                                     F1      IoU
-------------------------------------------------------
U-Net-3D (TS=6)                        0.375    0.338
SwinUNETR-3D (TS=6)                    0.374    0.331
UNETR-3D (TS=6)                        0.371    0.336
Att-U-Net-3D (TS=6)                    0.354    0.312
SwinUNETR-3D (TS=2)                    0.366    0.321
-------------------------------------------------------
Ours (TS=2)                            0.418    0.264

Delta: +0.043 (BEATS benchmark)

Total pipeline: 195.9min
Done.


In [11]:
# %% Cell 11: Per-fire aggregate visualization 
# Runs directly in this session using the trained model and in-memory FP stats.
# No separate weights/stats load needed.

from matplotlib.patches import Patch

VIS_DIR = os.path.join(CFG.SAVE_DIR, f"fp_v6_fire_maps{SFX}")
os.makedirs(VIS_DIR, exist_ok=True)

# Load best checkpoint (already saved during training)
model.load_state_dict(torch.load(os.path.join(CFG.SAVE_DIR, f"best_fp_v6{SFX}.pth"),
                                 weights_only=False))
model.eval()

def scale_bg(arr):
    bg = np.nan_to_num(arr, nan=0.0)
    pos = bg[bg > 0]
    if pos.size > 100:
        p2, p98 = np.percentile(pos, [2, 98])
    else:
        p2, p98 = bg.min(), bg.max()
    return np.clip((bg - p2) / (p98 - p2 + 1e-8), 0, 1)


def build_input_vis(fire_dir, input_days, ba_last):
    cs = CFG.CROP_SIZE; ts = CFG.TS_LENGTH
    frames = []
    for t_idx in range(ts):
        sat, fH, fW = load_frame_sat(fire_dir, input_days[t_idx])
        if fH < cs or fW < cs: return None
        sat_crop = center_crop(sat, cs, fH, fW)
        fp_raw = load_firepred_raw(fire_dir, input_days[t_idx])
        if fp_raw is None:
            fp_crop = np.zeros((len(CFG.FP_BAND_IDX), cs, cs), dtype=np.float32)
        else:
            fp_raw = np.nan_to_num(fp_raw, nan=0.0, posinf=0.0, neginf=0.0)
            fp_norm = normalize_firepred(fp_raw, CFG.FP_MEAN, CFG.FP_STD)
            fpH, fpW = fp_norm.shape[1], fp_norm.shape[2]
            if fpH < cs or fpW < cs:
                fp_crop = np.zeros((len(CFG.FP_BAND_IDX), cs, cs), dtype=np.float32)
            else:
                fp_crop = center_crop(fp_norm, cs, fpH, fpW)
        ba_t = ba_last if t_idx == ts - 1 else load_ba_mask(input_days[t_idx])
        if ba_t is not None:
            ba_crop = center_crop(ba_t, cs, fH, fW)[np.newaxis]
        else:
            ba_crop = np.zeros((1, cs, cs), dtype=np.float32)
        frames.append(np.concatenate([sat_crop, fp_crop, ba_crop], axis=0))
    return np.stack(frames, axis=1)


def run_fire_vis(fire_id, fire_dir, threshold):
    cs = CFG.CROP_SIZE; ts = CFG.TS_LENGTH
    day_files = sorted(glob.glob(os.path.join(fire_dir, "VIIRS_Day", "*.tif")))
    if len(day_files) < ts + 1: return None

    pred_union = np.zeros((cs, cs), dtype=np.float32)
    gt_union   = np.zeros((cs, cs), dtype=np.float32)
    bg_i4      = None
    n_windows = pair_ok = 0
    tp_sum = fp_sum = fn_sum = 0.0

    for i in range(len(day_files) - ts):
        input_days = day_files[i:i+ts]; label_day = day_files[i+ts]
        n_windows += 1
        ba_last = load_ba_mask(input_days[-1]); ba_next = load_ba_mask(label_day)
        if ba_last is None or ba_next is None: continue
        H, W = ba_last.shape
        if H < cs or W < cs: continue
        pair_ok += 1

        label_crop = center_crop(np.clip(ba_next - ba_last, 0, 1), cs, H, W)

        x = build_input_vis(fire_dir, input_days, ba_last)
        if x is None: continue
        x_t = torch.from_numpy(x).float().unsqueeze(0).to(device)
        with torch.no_grad(), autocast(enabled=CFG.USE_AMP):
            logits = model(x_t)
        pred = (torch.sigmoid(logits).squeeze().cpu().numpy() > threshold).astype(np.float32)

        tp_sum += float((pred * label_crop).sum())
        fp_sum += float((pred * (1 - label_crop)).sum())
        fn_sum += float(((1 - pred) * label_crop).sum())

        pred_union = np.maximum(pred_union, pred)
        gt_union   = np.maximum(gt_union, label_crop)

        d_label = read_tif(label_day)
        if d_label.shape[0] >= 4:
            bg_i4 = center_crop(d_label[3], cs, H, W)

    if pair_ok == 0: return None

    f1  = 2*tp_sum / (2*tp_sum + fp_sum + fn_sum + 1e-8)
    iou = tp_sum / (tp_sum + fp_sum + fn_sum + 1e-8)

    tp_map = pred_union * gt_union
    fp_map = pred_union * (1 - gt_union)
    fn_map = (1 - pred_union) * gt_union

    return {"fire_id": fire_id, "pair_ok": pair_ok, "n_windows": n_windows,
            "f1": f1, "iou": iou, "bg": bg_i4, "gt_union": gt_union,
            "tp_map": tp_map, "fp_map": fp_map, "fn_map": fn_map,
            "tp_sum": tp_sum, "fp_sum": fp_sum, "fn_sum": fn_sum}


def render_fire(res, path):
    cs = CFG.CROP_SIZE
    bg = scale_bg(res["bg"]) if res["bg"] is not None else np.zeros((cs, cs))
    rgb_base = np.stack([bg, bg, bg], axis=-1)

    # GT panel
    rgb_gt = rgb_base.copy()
    gtm = res["gt_union"] > 0.5
    rgb_gt[gtm] = rgb_gt[gtm] * 0.2 + np.array([1.0, 0.2, 0.2]) * 0.8

    # Prediction panel
    rgb_pr = rgb_base.copy()
    alpha = 0.85
    tpm = res["tp_map"] > 0.5
    fpm = res["fp_map"] > 0.5
    fnm = res["fn_map"] > 0.5
    rgb_pr[tpm] = rgb_pr[tpm] * (1 - alpha) + np.array([1.0, 0.0, 0.0]) * alpha
    rgb_pr[fpm] = rgb_pr[fpm] * (1 - alpha) + np.array([0.0, 0.8, 0.0]) * alpha
    rgb_pr[fnm] = rgb_pr[fnm] * (1 - alpha) + np.array([0.2, 0.3, 1.0]) * alpha

    fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))
    axes[0].imshow(rgb_gt); axes[0].axis("off")
    axes[0].set_title(f"Ground Truth (new-burn union)\n"
                      f"{int(res['gt_union'].sum())} px across {res['pair_ok']} windows",
                      fontsize=11)
    axes[1].imshow(rgb_pr); axes[1].axis("off")
    axes[1].set_title(f"Prediction (spatial union)\n"
                      f"F1={res['f1']:.3f}  IoU={res['iou']:.3f}  thr={best_thr:.2f}",
                      fontsize=11)
    legend = [Patch(facecolor="red", label="True Positive"),
              Patch(facecolor="green", label="False Positive"),
              Patch(facecolor="royalblue", label="False Negative")]
    fig.legend(handles=legend, loc="lower center", ncol=3, fontsize=10,
               frameon=True, bbox_to_anchor=(0.5, -0.02))
    fig.suptitle(res["fire_id"], fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close()


print(f"Rendering one map per fire to: {VIS_DIR}")
test_dirs = [(d, os.path.join(CFG.DATA_ROOT, d))
             for d in sorted(os.listdir(CFG.DATA_ROOT))
             if d.startswith("US_2021_")
             and os.path.isdir(os.path.join(CFG.DATA_ROOT, d, "VIIRS_Day"))]

fire_results = []
for i, (fid, fdir) in enumerate(test_dirs):
    res = run_fire_vis(fid, fdir, best_thr)
    if res is None:
        print(f"  [{i+1}/{len(test_dirs)}] {fid}: skipped"); continue
    fire_results.append(res)
    render_fire(res, os.path.join(VIS_DIR, f"{fid}.png"))
    print(f"  [{i+1}/{len(test_dirs)}] {fid}: F1={res['f1']:.3f} IoU={res['iou']:.3f}")

# Aggregate and sanity-check vs Cell 9 test result
tot_tp = sum(r["tp_sum"] for r in fire_results)
tot_fp = sum(r["fp_sum"] for r in fire_results)
tot_fn = sum(r["fn_sum"] for r in fire_results)
vis_agg_f1  = 2*tot_tp / (2*tot_tp + tot_fp + tot_fn + 1e-8)
vis_agg_iou = tot_tp / (tot_tp + tot_fp + tot_fn + 1e-8)
print(f"\nVis aggregate: F1={vis_agg_f1:.4f} IoU={vis_agg_iou:.4f}")
print(f"Cell 9 test:   F1={agg_f1:.4f} IoU={agg_iou:.4f}")
print(f"Match: {'YES' if abs(vis_agg_f1 - agg_f1) < 1e-3 else 'NO (small diff expected if any window-skip differs)'}")

# Per-fire summary bar chart
fig, ax = plt.subplots(figsize=(max(8, len(fire_results)*0.5), 5))
ids_short = [r["fire_id"].replace("US_2021_", "") for r in fire_results]
f1s = [r["f1"] for r in fire_results]
ax.bar(range(len(f1s)), f1s,
       color=["#e74c3c" if f >= 0.375 else "#4a86c8" for f in f1s])
ax.axhline(0.375, color="k", ls="--", lw=1, label="Paper best F1 (0.375)")
ax.axhline(vis_agg_f1, color="green", ls=":", lw=1, label=f"Aggregate F1 ({vis_agg_f1:.3f})")
ax.set_xticks(range(len(f1s)))
ax.set_xticklabels(ids_short, rotation=60, ha="right", fontsize=7)
ax.set_ylabel("F1"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3, axis="y")
ax.set_title("Per-fire F1 (red bars beat paper best)")
plt.tight_layout()
plt.savefig(os.path.join(VIS_DIR, "_summary_per_fire_f1.png"), dpi=150, bbox_inches="tight")
plt.close()
print(f"Summary: {os.path.join(VIS_DIR, '_summary_per_fire_f1.png')}")
print("Done.")


Rendering one map per fire to: /kaggle/working/fp_v6_fire_maps_s44
  [1/24] US_2021_AZ3345510938920210616: F1=0.000 IoU=0.000
  [2/24] US_2021_AZ3368910927620210616: F1=0.004 IoU=0.002
  [3/24] US_2021_CA3451712013120211011: F1=0.366 IoU=0.224
  [4/24] US_2021_CA3568711855020210818: F1=0.438 IoU=0.280
  [5/24] US_2021_CA3604711863120210910: F1=0.353 IoU=0.215
  [6/24] US_2021_CA3627811855020210815: F1=0.221 IoU=0.124
  [7/24] US_2021_CA3658211879520210912: F1=0.430 IoU=0.274
  [8/24] US_2021_CA4086312235520210630: F1=0.052 IoU=0.026
  [9/24] US_2021_FL2521008104520210308: F1=0.000 IoU=0.000
  [10/24] US_2021_ID4453211532920210810: F1=0.422 IoU=0.267
  [11/24] US_2021_ID4558511544420210705: F1=0.451 IoU=0.291
  [12/24] US_2021_ID4663811466720210707: F1=0.457 IoU=0.296
  [13/24] US_2021_ID4762711608320210708: F1=0.278 IoU=0.161
  [14/24] US_2021_MT4568311385420210708: F1=0.460 IoU=0.298
  [15/24] US_2021_MT4579011310120210708: F1=0.399 IoU=0.249
  [16/24] US_2021_MT4714310953420211004: F